# KLDM Pipeline

This notebook walks through the KLDM data pipeline in two parts:

1. **Data loading**: MP-20 raw CSVs, MatterGen cache building, `ChemGraph` samples, and KLDM transforms.
2. **Algorithm 1: training targets**: continuous diffusion on one-hot atom features, lattice diffusion, and trivialised diffusion on velocity.

The goal is to make each step visible and intuitive, not just produce tensors.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch
from torch_geometric.data import Batch

ROOT = Path.cwd()
SRC_ROOT = ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from kldm.data.dataset import MP20
from kldm.data.tasks import DNGTask, CSPTask
from kldm.data.transformations import training_transform
from kldm.kldm import ModelKLDM

DATA_ROOT = ROOT / "src" / "kldm" / "data" / "data" / "mp_20_notebook"
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"

print("DATA_ROOT:", DATA_ROOT)
print("raw exists:", RAW_ROOT.exists())
print("processed exists:", PROCESSED_ROOT.exists())

/workspace/.venv/lib/python3.11/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


DATA_ROOT: /workspace/src/kldm/data/data/mp_20_notebook
raw exists: True
processed exists: True


## Section 1. MP-20 data loading, MatterGen cache, and KLDM transforms

We start from the raw MP-20 CSV files, then inspect a MatterGen-backed `ChemGraph`, and finally show how KLDM transforms that sample into the tensors used by the model.

In [2]:
# Preview the raw MP-20 CSV that backs the dataset.
raw_preview = pd.read_csv(RAW_ROOT / "train.csv", nrows=3)
raw_preview[[col for col in raw_preview.columns if col != "cif"]]

,Unnamed: 0.1,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,spacegroup.number,spacegroup.number.conv,cif.conv
0,0,37228,mp-1221227,-1.637460,0.2133,Na3MnCoNiO6,0.043001,"['Co', 'Mn', 'Na', 'Ni', 'O']",8,8,# generated using pymatgen\ndata_Na3MnCoNiO6\n...
1,1,19480,mp-974729,-0.314759,0.0000,Nd(Al2Cu)4,0.000000,"['Al', 'Cu', 'Nd']",139,139,# generated using pymatgen\ndata_Nd(Al2Cu)4\n_...
2,2,29624,mp-1185360,-0.193761,0.0000,LiMnIr2,0.018075,"['Ir', 'Li', 'Mn']",225,225,# generated using pymatgen\ndata_LiMnIr2\n_sym...


In [3]:
# Build the KLDM dataset wrapper without any transform.
# Under the hood this uses MatterGen's cache + ChemGraph dataset.
raw_dataset = MP20(DATA_ROOT, split="train")
print("dataset length:", len(raw_dataset))
print("raw folder:", raw_dataset.raw_folder)
print("processed folder:", raw_dataset.processed_folder)

MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
dataset length: 8
raw folder: /workspace/src/kldm/data/data/mp_20_notebook/raw
processed folder: /workspace/src/kldm/data/data/mp_20_notebook/processed


In [4]:
# Inspect the underlying MatterGen sample directly.
chemgraph = raw_dataset.data[0]

print("MatterGen sample type:", type(chemgraph))
print("Available keys:", sorted(chemgraph.keys()))
print("pos shape:", tuple(chemgraph.pos.shape))
print("atomic_numbers shape:", tuple(chemgraph.atomic_numbers.shape))
print("cell shape:", tuple(chemgraph.cell.shape))
print("num_atoms:", chemgraph.num_atoms.item())

MatterGen sample type: <class 'mattergen.common.data.chemgraph.ChemGraph'>
Available keys: ['atomic_numbers', 'cell', 'dft_band_gap', 'energy_above_hull', 'formation_energy_per_atom', 'num_atoms', 'num_nodes', 'pos', 'space_group']
pos shape: (12, 3)
atomic_numbers shape: (12,)
cell shape: (1, 3, 3)
num_atoms: 12


In [16]:
# The KLDM wrapper converts that ChemGraph into a PyG Data sample.
plain_sample = raw_dataset[0]

print("KLDM sample type:", type(plain_sample))
print("pos:", tuple(plain_sample.pos.shape))
print("h:", tuple(plain_sample.h.shape))
print("lengths:", tuple(plain_sample.lengths.shape))
print("angles:", tuple(plain_sample.angles.shape))

edge_node_index = getattr(plain_sample, "edge_node_index", None)
edge_index = getattr(plain_sample, "edge_index", None)
if edge_node_index is not None:
    print("edge_node_index:", tuple(edge_node_index.shape))
elif edge_index is not None:
    print("edge_index:", tuple(edge_index.shape))
else:
    print("No concrete edge index tensor on plain_sample.")
    print("keys:", list(plain_sample.keys()))

plain_sample

KLDM sample type: <class 'torch_geometric.data.data.Data'>
pos: (12, 3)
h: (12,)
lengths: (1, 3)
angles: (1, 3)
No concrete edge index tensor on plain_sample.
keys: ['formation_energy_per_atom', 'h', 'lengths', 'space_group', 'dft_band_gap', 'pos', 'angles', 'energy_above_hull']


Data(pos=[12, 3], h=[12], lengths=[1, 3], angles=[1, 3], space_group=[1], formation_energy_per_atom=[1], dft_band_gap=[1], energy_above_hull=[1])

In [17]:
# Apply the KLDM training transform for DNG-style atom handling.
# This does three important things:
# 1. keeps a complete graph in edge_node_index
# 2. turns lengths/angles into KLDM's lattice representation
# 3. one-hot encodes the atomic numbers
transformed_dataset = MP20(
    DATA_ROOT,
    split="train",
    transform=training_transform(species_mode="one_hot"),
)
transformed_sample = transformed_dataset[0]

print("Before transform")
print("  h shape:", tuple(plain_sample.h.shape))
print("  h[:5]:", plain_sample.h[:5])
print("  lengths[0]:", plain_sample.lengths[0])
print("  angles[0]:", plain_sample.angles[0])

print("\nAfter transform")
print("  h shape:", tuple(transformed_sample.h.shape))
print("  h[0]:", transformed_sample.h[0])
print("  lengths[0]:", transformed_sample.lengths[0])
print("  angles[0]:", transformed_sample.angles[0])
print("  l[0]:", transformed_sample.l[0])

Before transform
  h shape: (12,)
  h[:5]: tensor([11, 11, 11, 25, 27])
  lengths[0]: tensor([3.0293, 5.6371, 7.9776])
  angles[0]: tensor([107.5114, 100.9447,  90.0000])

After transform
  h shape: (12, 118)
  h[0]: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
  lengths[0]: tensor([1.1083, 1.7294, 2.0766])
  angles[0]: tensor([0.3155, 0.1934, 0.0000])
  l[0]: tensor([1.1083, 1.7294, 2.0766, 0.3155, 0.1934, 0.0000])


In [11]:
# The Task wrappers add KLDM-specific metadata on top of the transformed sample.
dng = DNGTask()
csp = CSPTask()

dng_sample = dng.fit_dataset(DATA_ROOT, split="train")[0]
csp_sample = csp.fit_dataset(DATA_ROOT, split="train")[0]

print("DNG metadata")
print("  task_id:", dng_sample.task_id.item())
print("  diffuse_h:", bool(dng_sample.diffuse_h.item()))
print("  is_prior:", bool(dng_sample.is_prior.item()))

print("\nCSP metadata")
print("  task_id:", csp_sample.task_id.item())
print("  diffuse_h:", bool(csp_sample.diffuse_h.item()))
print("  is_prior:", bool(csp_sample.is_prior.item()))

DNG metadata
  task_id: 1
  diffuse_h: True
  is_prior: False

CSP metadata
  task_id: 0
  diffuse_h: False
  is_prior: False


## Section 2. Algorithm 1: KLDM training targets

Now we move from the dataloader to **Algorithm 1**.

For a batch of transformed crystals, KLDM builds three perturbation targets:

1. **Atom types**: continuous diffusion on one-hot encoded atoms.
2. **Lattice**: variance-preserving diffusion on the 6D lattice vector.
3. **Velocity**: trivialised diffusion in tangent space.

We will compare the clean inputs with the perturbed targets step by step.

In [18]:
# Build a small DNG-style batch with one-hot atoms.
smoke_dataset = dng.fit_dataset(DATA_ROOT, split="train")
smoke_batch = Batch.from_data_list([smoke_dataset[0], smoke_dataset[1]])
model = ModelKLDM()

print("smoke_batch.pos:", tuple(smoke_batch.pos.shape))
print("smoke_batch.h:", tuple(smoke_batch.h.shape))
print("smoke_batch.lengths:", tuple(smoke_batch.lengths.shape))
print("smoke_batch.angles:", tuple(smoke_batch.angles.shape))
print("num_graphs:", smoke_batch.num_graphs)

smoke_batch.pos: (25, 3)
smoke_batch.h: (25, 118)
smoke_batch.lengths: (2, 3)
smoke_batch.angles: (2, 3)
num_graphs: 2


In [20]:
# Generate Algorithm 1 targets.
targets = model.training_targets(smoke_batch)
sorted(targets.keys())

['a_t',
 'eps_a',
 'eps_l',
 'eps_v',
 'l_t',
 'sigma_v',
 'target_a',
 'target_l',
 'target_v',
 'v_t']

In [21]:
# 2.1 Continuous diffusion on one-hot encoded atoms.
print("Original atom representation")
print("  smoke_batch.h shape:", tuple(smoke_batch.h.shape))
print("  smoke_batch.h[0]:", smoke_batch.h[0])

if "a_t" in targets:
    print("\nAfter atom diffusion")
    print("  a_t shape:", tuple(targets["a_t"].shape))
    print("  a_t[0]:", targets["a_t"][0])
    print("  target_a[0]:", targets["target_a"][0])
    print("  eps_a[0]:", targets["eps_a"][0])
else:
    print("Atom diffusion targets were not produced for this batch.")

Original atom representation
  smoke_batch.h shape: (25, 118)
  smoke_batch.h[0]: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

After atom diffusion
  a_t shape: (25, 118)
  a_t[0]: tensor([ 0.1798,  0.1495, -1.1839,  0.0057,  1.1767, -0.1718, -0.6781, -0.1642,
         0.3591,  1.0523,  0.5487,  0.9527,  1.0286,  0.5603, -0.3085, -0.1207,
         0.3336,  0.6064, -0.3947, -0.5408,  0.5911,  0.5824, -1.1847,  0.4285,
         0.3363, -0.3739,  0.3702,  0.2681,  0.4106, -0.5247, -0.4588,  0.0261,
        -0.606

In [22]:
# 2.2 Lattice diffusion on the 6D KLDM lattice vector.
clean_lattice = torch.cat([smoke_batch.lengths[0], smoke_batch.angles[0]])

print("Original lattice")
print("  lengths[0]:", smoke_batch.lengths[0])
print("  angles[0]:", smoke_batch.angles[0])
print("  lattice vector[0]:", clean_lattice)

print("\nAfter lattice diffusion")
print("  l_t[0]:", targets["l_t"][0])
print("  target_l[0]:", targets["target_l"][0])
print("  eps_l[0]:", targets["eps_l"][0])

Original lattice
  lengths[0]: tensor([1.1083, 1.7294, 2.0766])
  angles[0]: tensor([0.3155, 0.1934, 0.0000])
  lattice vector[0]: tensor([1.1083, 1.7294, 2.0766, 0.3155, 0.1934, 0.0000])

After lattice diffusion
  l_t[0]: tensor([ 0.7122,  1.8756,  2.1932,  1.1367, -0.0364, -0.2869])
  target_l[0]: tensor([-0.1640,  0.8839,  0.9734,  1.3459, -0.2682, -0.4278])
  eps_l[0]: tensor([-0.1640,  0.8839,  0.9734,  1.3459, -0.2682, -0.4278])


In [23]:
# 2.3 Trivialised diffusion on velocity.
# KLDM starts from v0 = 0 by design, then perturbs in tangent space.
v0 = model._extract_velocities(smoke_batch)

print("Original velocity-related state")
print("  pos[0:3]:")
print(smoke_batch.pos[:3])
print("  v0[0:3]:")
print(v0[:3])

print("\nAfter trivialised diffusion")
print("  v_t[0:3]:")
print(targets["v_t"][:3])
print("  target_v[0:3]:")
print(targets["target_v"][:3])
print("  eps_v[0:3]:")
print(targets["eps_v"][:3])
print("  sigma_v[0:3]:")
print(targets["sigma_v"][:3])

Original velocity-related state
  pos[0:3]:
tensor([[6.6651e-01, 1.7850e-03, 3.3302e-01],
        [1.2300e-04, 9.9972e-01, 2.4600e-04],
        [3.3202e-01, 9.9566e-01, 6.6405e-01]])
  v0[0:3]:
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])

After trivialised diffusion
  v_t[0:3]:
tensor([[ 0.2271, -1.3259, -0.4496],
        [ 0.5860,  0.7975, -0.0797],
        [-0.4350,  0.6276, -0.3046]])
  target_v[0:3]:
tensor([[-0.5050,  2.9482,  0.9997],
        [-1.3030, -1.7733,  0.1772],
        [ 0.9672, -1.3955,  0.6774]])
  eps_v[0:3]:
tensor([[ 0.3387, -1.9771, -0.6704],
        [ 0.8738,  1.1892, -0.1189],
        [-0.6486,  0.9358, -0.4543]])
  sigma_v[0:3]:
tensor([[0.6706],
        [0.6706],
        [0.6706]])


In [24]:
# Compact shape summary for the full Algorithm 1 output.
summary = {
    "input_pos": tuple(smoke_batch.pos.shape),
    "input_h": tuple(smoke_batch.h.shape),
    "input_lengths": tuple(smoke_batch.lengths.shape),
    "input_angles": tuple(smoke_batch.angles.shape),
}
summary.update({
    key: tuple(value.shape)
    for key, value in targets.items()
    if torch.is_tensor(value)
})
summary

{'input_pos': (25, 3),
 'input_h': (25, 118),
 'input_lengths': (2, 3),
 'input_angles': (2, 3),
 'v_t': (25, 3),
 'target_v': (25, 3),
 'eps_v': (25, 3),
 'sigma_v': (25, 1),
 'l_t': (2, 6),
 'target_l': (2, 6),
 'eps_l': (2, 6),
 'a_t': (25, 118),
 'target_a': (25, 118),
 'eps_a': (25, 118)}